In [1]:
import torch

In [ ]:
from __future__ import annotations
from typing import Dict, List, Optional, Literal, Tuple
import numpy as np
import pandas as pd

# --- Dose dictionary (mg per g) ---
lenuzza_doses_mg_per_g: Dict[str, float] = {
    "memantine": 0.005,
    "omeprazole": 0.010,
    "repaglinide": 0.00025,
    "rosuvastatin": 0.005,
    "tolbutamide": 0.010,
    "dextromethorphan": 0.018,
    "digoxin": 0.00025,
    "paracetamol": 0.060,
    "caffeine": 0.073,
    "midazolam": 0.004,
    "paraxanthine": 0.073,
    "dextrorphan": 0.018,
}

def _infer_dose_for_substance(substance_name: str) -> float:
    s = (substance_name or "").lower()
    for k, v in lenuzza_doses_mg_per_g.items():
        if k in s:
            return v
    # fallback if not matched
    return 0.5

def _build_individual_json(
    subject_name: str,
    subdf: pd.DataFrame,
    *,
    normalize_time: bool,
    dosing_route_default: str,
) -> Dict:
    """
    subdf: rows for a single subject within a single (study_name, substance_label).
           Must contain columns: 'time', 'value'.
           Optional: 'dosing_time', 'dosing_value', 'dosing_type', 'substance_name'
    """
    # Sort and sanitize observations
    g = subdf.sort_values("time")
    t = g["time"].astype(float).to_numpy()
    y = g["value"].astype(float).to_numpy()

    # Drop NaNs consistently (on either, mask both)
    valid = np.isfinite(t) & np.isfinite(y)
    t, y = t[valid], y[valid]

    # Optional per-substance normalization [0,1] (no time shift if degenerate)
    if normalize_time and len(t) > 0:
        tmin, tmax = float(np.min(t)), float(np.max(t))
        if tmax > tmin:
            t_norm = (t - tmin) / (tmax - tmin)
        else:
            t_norm = np.zeros_like(t)
        times_out = t_norm.tolist()
    else:
        times_out = t.tolist()

    # Dosing info: prefer explicit columns if present, otherwise infer single dose @ t=0
    if {"dosing_times", "dosing", "dosing_type", "dosing_name"}.issubset(set(g.columns)):
        # Already denormalized fields present (rare). Expect list-likes per row; collapse.
        # If not present as lists, we’ll reconstruct from scalar columns below.
        raise ValueError(
            "Found dosing_* wide columns unexpectedly; this loader expects tidy rows. "
            "Provide columns 'dosing_time', 'dosing_value', 'dosing_type', 'substance_name' per dose event, "
            "or omit them and we’ll infer a single dose."
        )

    # If tidy dosing columns exist, collect; otherwise infer one oral dose at t=0
    dosing_times, dosing_vals, dosing_types, dosing_names = [], [], [], []
    has_tidy_dose_cols = all(c in g.columns for c in ["dosing_time", "dosing_value"])
    if has_tidy_dose_cols:
        # Filter finite entries
        dmask = np.isfinite(g["dosing_time"]) & np.isfinite(g["dosing_value"])
        if dmask.any():
            d_times = g.loc[dmask, "dosing_time"].astype(float).to_numpy()
            d_vals = g.loc[dmask, "dosing_value"].astype(float).to_numpy()
            # Optional normalization of dosing times must match observation normalization
            if normalize_time and len(t) > 0:
                # Use same (tmin,tmax) to maintain temporal consistency
                tmin, tmax = float(np.min(t)), float(np.max(t))
                if tmax > tmin:
                    d_times = (d_times - tmin) / (tmax - tmin)
                else:
                    d_times = np.zeros_like(d_times)
            dosing_times = d_times.tolist()
            dosing_vals = d_vals.tolist()
            # fallbacks for names/types
            d_type = g["dosing_type"].iloc[0] if "dosing_type" in g.columns else dosing_route_default
            s_name = g["substance_name"].iloc[0] if "substance_name" in g.columns else g["substance_label"].iloc[0]
            dosing_types = [str(d_type)] * len(dosing_times)
            dosing_names = [str(s_name)] * len(dosing_times)
    if not dosing_times:
        # Infer a single oral dose event
        s_name = (
            str(g["substance_name"].iloc[0]) if "substance_name" in g.columns
            else str(g["substance_label"].iloc[0])
        )
        inferred_dose = _infer_dose_for_substance(s_name)
        dosing_times = [0.0 if not normalize_time else 0.0]
        dosing_vals = [float(inferred_dose)]
        dosing_types = [dosing_route_default]
        dosing_names = [s_name]

    # Fields required by your schema but not present from CSV (use empty lists)
    remaining = []
    remaining_times = []
    covariates: List[Dict] = []

    return {
        "name_id": str(subject_name),
        "observations": [float(v) for v in y.tolist()],
        "times": [float(tt) for tt in times_out],
        "remaining": remaining,
        "remaining_times": remaining_times,
        "dosing": [float(d) for d in dosing_vals],
        "dosing_type": [str(dt) for dt in dosing_types],
        "dosing_times": [float(dt) for dt in dosing_times],
        "dosing_name": [str(dn) for dn in dosing_names],
        "covariates": covariates,
    }

def _split_context_target(
    individuals: List[Dict],
    strategy: Literal["leave_one_out", "random_fraction"] = "leave_one_out",
    *,
    rng: Optional[np.random.Generator] = None,
    context_fraction: float = 0.5,
) -> List[Tuple[List[Dict], List[Dict]]]:
    """
    Returns a list of (context, target) pairs.
    - leave_one_out: yields N study_jsons by holding out each subject once.
    - random_fraction: yields a single split with ~context_fraction in context.
    """
    N = len(individuals)
    if N == 0:
        return [([], [])]
    if strategy == "leave_one_out":
        pairs = []
        for i in range(N):
            target = [individuals[i]]
            context = [ind for j, ind in enumerate(individuals) if j != i]
            pairs.append((context, target))
        return pairs
    elif strategy == "random_fraction":
        rng = rng or np.random.default_rng(0)
        idx = np.arange(N)
        rng.shuffle(idx)
        k = max(1, int(round(context_fraction * N)))
        context_idx = set(idx[:k])
        context = [individuals[i] for i in range(N) if i in context_idx]
        target = [individuals[i] for i in range(N) if i not in context_idx]
        if len(target) == 0:
            # ensure at least one target
            context, target = context[:-1], context[-1:]
        return [(context, target)]
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

def dataframe_to_study_jsons(
    df: pd.DataFrame,
    *,
    normalize_time: bool = True,
    dosing_route_default: str = "oral",
    split_strategy: Literal["leave_one_out", "random_fraction"] = "leave_one_out",
    context_fraction: float = 0.5,
    rng: Optional[np.random.Generator] = None,
) -> List[Dict]:
    """
    Convert the tidy CSV into a list of study_json dicts:
        {
          "context": [individual_json, ...],
          "target":  [individual_json, ...],
          "meta_data": {"study_name": str, "substance_name": str}
        }

    Expected minimum columns in df:
      - 'study_name', 'substance_label', 'subject_name', 'time', 'value'
    Optional dosing columns (tidy per event):
      - 'dosing_time', 'dosing_value', ['dosing_type'], ['substance_name']
    """
    required = {"study_name", "substance_label", "subject_name", "time", "value"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Input DataFrame missing required columns: {sorted(missing)}")

    study_jsons: List[Dict] = []

    # Group by (study_name, substance_label)
    for (study_name, substance_label), g_study in df.groupby(["study_name", "substance_label"], sort=False):
        # Build per-subject individuals
        individuals: List[Dict] = []
        for subject_name, g_subj in g_study.groupby("subject_name", sort=False):
            ind = _build_individual_json(
                subject_name,
                g_subj,
                normalize_time=normalize_time,
                dosing_route_default=dosing_route_default,
            )
            individuals.append(ind)

        # Produce (context, target) pairs
        pairs = _split_context_target(
            individuals,
            strategy=split_strategy,
            rng=rng,
            context_fraction=context_fraction,
        )

        # One study_json per split
        for context, target in pairs:
            study_jsons.append({
                "context": context,
                "target": target,
                "meta_data": {
                    "study_name": str(study_name),
                    "substance_name": str(substance_label),
                },
            })

    return study_jsons

In [4]:
import os
from pff import data_dir

test_dataset_path =("preprocessed", "lenuzza", "Lenuzza2016.csv")
csv_path = os.path.join(
        data_dir, *test_dataset_path
    )

In [ ]:
df = pd.read_csv(csv_path)

In [6]:
dataframe_to_study_jsons(df)

[{'context': [{'name_id': 'memantine_4',
    'observations': [2.2672625000000002e-07,
     6.734340000000001e-06,
     5.859163000000001e-06,
     6.092060600000001e-06,
     4.902836300000001e-06,
     3.1755120000000003e-06,
     2.9048233000000004e-06,
     1.1719898000000002e-06],
    'times': [0.0,
     0.009009009009009009,
     0.03903903903903904,
     0.06306306306306306,
     0.13513513513513514,
     0.27927927927927926,
     0.42342342342342343,
     1.0],
    'remaining': [],
    'remaining_times': [],
    'dosing': [0.005],
    'dosing_type': ['oral'],
    'dosing_times': [0.0],
    'dosing_name': ['memantine'],
    'covariates': []},
   {'name_id': 'memantine_8',
    'observations': [6.678324000000001e-07,
     5.635168600000001e-06,
     5.419737300000001e-06,
     5.183596600000001e-06,
     5.300352000000001e-06,
     3.4714255000000005e-06,
     2.7019715000000005e-06,
     5.198049000000001e-07],
    'times': [0.0,
     0.009009009009009009,
     0.03903903903903904